# 31 — B1 DEV diagnostic + leader handoff (CPU, fast)

Notebook này không chạy lại retrieval hay model. Nó xác minh đầu ra notebook
30, lấy đúng các query có điểm thay đổi, nối chúng với ground truth DEV và các
candidate B1 nằm trong vùng đáp án, rồi xuất một decision packet có thể gửi
leader.

Kết quả notebook 30 là tín hiệu dương nhưng chưa chắc chắn: B0 `0.5195767`, B1
`0.5291005`, chênh `+0.0095238`, bootstrap 95% cắt qua 0. Notebook 31 không
biến kết quả đó thành bằng chứng mạnh hơn; nó chỉ làm rõ bằng chứng hiện có.

Chạy bằng CPU, bật Internet và Secrets GitHub/Modal. Không cần Add Input
`mfe-data-v2`, không mở held-out và không sửa BTC.


## 0 — Khóa diagnostic


In [ ]:
from pathlib import Path

RUN_SCOPE = "DEV_DIAGNOSTIC"
CONFIRM_RUN = "B1_HANDOFF"
EXPECTED_B1_COUNT = 5301
EXPECTED_DEV_QUERIES = 63
EXPECTED_CHANGED = 3
EXPECTED_RESCUED = 2
EXPECTED_REGRESSED = 1
EXPECTED_B0_MEAN = 0.5195767195767196
EXPECTED_B1_MEAN = 0.5291005291005292
EXPECTED_MEAN_DELTA = 0.009523809523809523
EXPECTED_BOOTSTRAP_LOWER = -0.006349206349206349
EXPECTED_BOOTSTRAP_UPPER = 0.028650793650792497
EXPECTED_NB29_COMPLETE_SHA256 = "d2c38a6a5f7fe810b09d87b6a384b850bcbf942564bf83d910c353eaeafc5fa6"

REPO_URL = "https://github.com/AIVIETNAM-AIO-Triet-Descartes/multimodal-fusion-engine.git"
REPO_REF = "feature/quang/storage"
REPO = Path("/kaggle/working/multimodal-fusion-engine")
WORK = Path("/kaggle/working/b1-dev-diagnostic")
INPUT = WORK / "input"
OUTPUT = WORK / "output"

if RUN_SCOPE != "DEV_DIAGNOSTIC" or CONFIRM_RUN != "B1_HANDOFF":
    raise RuntimeError("Diagnostic lock is closed")
for path in (WORK, INPUT, OUTPUT):
    path.mkdir(parents=True, exist_ok=True)


## 1 — Helpers


In [ ]:
from __future__ import annotations

from typing import Any


def signed_payload(payload: dict[str, Any], field: str, hash_json) -> dict[str, Any]:
    unsigned = dict(payload)
    claimed = unsigned.pop(field, None)
    if not isinstance(claimed, str) or claimed != hash_json(unsigned):
        raise RuntimeError(f"Invalid signed payload: {field}")
    return unsigned


def query_intervals(query: dict[str, Any], *, strict: bool) -> list[tuple[int, int]]:
    if query["type"] == "TRAKE":
        key = "gt_events_strict" if strict and query.get("gt_events_strict") else "gt_events"
        return [(int(start), int(end)) for start, end in query[key]]
    key = "gt_frame_range_strict" if strict and query.get("gt_frame_range_strict") else "gt_frame_range"
    start, end = query[key]
    return [(int(start), int(end))]


def candidate_memberships(query: dict[str, Any], candidates: list[dict[str, Any]]) -> dict[str, Any]:
    video_id = str(query["gt_video_id"])
    in_video = [row for row in candidates if str(row["video_id"]) == video_id]
    wide = query_intervals(query, strict=False)
    strict_ranges = query_intervals(query, strict=True)

    def matches(ranges: list[tuple[int, int]]) -> list[dict[str, Any]]:
        return [
            {
                "keyframe_id": str(row["keyframe_id"]),
                "frame_idx": int(row["frame_idx"]),
                "matching_events": [
                    index for index, (start, end) in enumerate(ranges)
                    if start <= int(row["frame_idx"]) <= end
                ],
            }
            for row in in_video
            if any(start <= int(row["frame_idx"]) <= end for start, end in ranges)
        ]

    return {
        "b1_candidates_in_video": len(in_video),
        "b1_in_wide_gt": matches(wide),
        "b1_in_strict_gt": matches(strict_ranges),
    }


## 2 — Repo và research volume


In [ ]:
import json
import math
import os
import subprocess
import sys
from datetime import datetime, timezone


def run(command: list[str], *, cwd: Path | None = None, env: dict[str, str] | None = None) -> str:
    result = subprocess.run(command, cwd=cwd, env=env, check=False, text=True, capture_output=True)
    if result.returncode:
        raise RuntimeError(f"Command failed: {' '.join(command[:4])}\n{result.stderr[-1600:]}")
    return result.stdout.strip()


from kaggle_secrets import UserSecretsClient

secrets = UserSecretsClient()
github_token = secrets.get_secret("GITHUB_TOKEN")
os.environ["MODAL_TOKEN_ID"] = secrets.get_secret("MODAL_TOKEN_ID")
os.environ["MODAL_TOKEN_SECRET"] = secrets.get_secret("MODAL_TOKEN_SECRET")
askpass = WORK / "git-askpass.sh"
askpass.write_text('#!/bin/sh\ncase "$1" in *Username*) printf x-access-token ;; *) printf %s "$GITHUB_TOKEN" ;; esac\n', encoding="utf-8")
askpass.chmod(0o700)
git_env = os.environ.copy()
git_env.update({"GITHUB_TOKEN": github_token, "GIT_ASKPASS": str(askpass), "GIT_TERMINAL_PROMPT": "0"})
try:
    if (REPO / ".git").is_dir():
        if run(["git", "status", "--porcelain"], cwd=REPO):
            raise RuntimeError("Restart Session: Kaggle checkout has local changes")
        run(["git", "fetch", "--depth", "1", "origin", REPO_REF], cwd=REPO, env=git_env)
        run(["git", "checkout", "--detach", "FETCH_HEAD"], cwd=REPO)
    else:
        run(["git", "clone", "--depth", "1", "--branch", REPO_REF, REPO_URL, str(REPO)], env=git_env)
finally:
    askpass.unlink(missing_ok=True)

run([sys.executable, "-m", "pip", "install", "-q", "pydantic>=2.6,<3", "pyyaml>=6,<7", "modal>=1,<2"])
sys.path.insert(0, str(REPO / "src"))

import yaml

from mfe.offline.augmentation.io import canonical_hash, read_jsonl, sha256_file, write_json_atomic, write_jsonl_atomic
from mfe.offline.transport import ModalVolumeTransport

repo_commit = run(["git", "rev-parse", "HEAD"], cwd=REPO)
cfg = yaml.safe_load((REPO / "config/keyframe_augmentation_kaggle.yaml").read_text(encoding="utf-8"))
research = ModalVolumeTransport(cfg["output"]["volume"], create_if_missing=False)
root = cfg["output"]["root"].rstrip("/")
nb29_root = f"{root}/gate1-b1-feature-aggregate-v1"
nb30_root = f"{root}/gate1-b0-b1-dev-ab-v1"
output_root = f"{root}/gate1-b1-dev-diagnostic-v1"
print({"repo_commit": repo_commit, "scope": RUN_SCOPE, "device": "CPU", "heldout_accessed": False})


## 3 — Xác minh notebook 30 và lấy đúng ba query thay đổi


In [ ]:
complete_path = INPUT / "nb30-COMPLETE.json"
research.get(f"{nb30_root}/COMPLETE.json", complete_path)
complete = json.loads(complete_path.read_text(encoding="utf-8"))
signed_payload(complete, "manifest_payload_hash", canonical_hash)
checks = {
    "status": complete.get("status") == "COMPLETE",
    "output_status": complete.get("output_status") == "DEV_POSITIVE_BUT_UNCERTAIN",
    "query_count": complete.get("query_count") == EXPECTED_DEV_QUERIES,
    "b0_mean": math.isclose(float(complete.get("b0_mean", -1)), EXPECTED_B0_MEAN, abs_tol=1e-12),
    "b1_mean": math.isclose(float(complete.get("b1_mean", -1)), EXPECTED_B1_MEAN, abs_tol=1e-12),
    "mean_delta": math.isclose(float(complete.get("mean_delta", -1)), EXPECTED_MEAN_DELTA, abs_tol=1e-12),
    "nb29": complete.get("nb29_complete_sha256") == EXPECTED_NB29_COMPLETE_SHA256,
}
failed = [name for name, passed in checks.items() if not passed]
if failed:
    raise RuntimeError(f"Notebook 30 result drift: {failed}")

ledger = complete.get("artifact_ledger", {})
if canonical_hash(ledger) != complete.get("artifact_ledger_sha256"):
    raise RuntimeError("Notebook 30 artifact ledger drift")
downloaded = {}
for name, spec in sorted(ledger.items()):
    local = INPUT / name
    remote = f"{nb30_root}/{name}"
    research.get(remote, local)
    if local.stat().st_size != int(spec["bytes"]):
        raise RuntimeError(f"Notebook 30 artifact size mismatch: {name}")
    if sha256_file(local) != spec["sha256"] or research.sha256(remote) != spec["sha256"]:
        raise RuntimeError(f"Notebook 30 artifact SHA mismatch: {name}")
    downloaded[name] = local

report = json.loads(downloaded["dev-ab-report.json"].read_text(encoding="utf-8"))
signed_payload(report, "report_payload_hash", canonical_hash)
per_query = list(read_jsonl(downloaded["per-query-dev.jsonl"]))
if len(per_query) != EXPECTED_DEV_QUERIES:
    raise RuntimeError("Notebook 30 per-query count drift")
changed = [row for row in per_query if not math.isclose(float(row["delta"]), 0.0, abs_tol=1e-15)]
rescued = [row for row in changed if float(row["delta"]) > 0]
regressed = [row for row in changed if float(row["delta"]) < 0]
if (len(changed), len(rescued), len(regressed)) != (EXPECTED_CHANGED, EXPECTED_RESCUED, EXPECTED_REGRESSED):
    raise RuntimeError("Changed/rescued/regressed query count drift")
if not math.isclose(float(report["bootstrap_95"]["lower"]), EXPECTED_BOOTSTRAP_LOWER, abs_tol=1e-12):
    raise RuntimeError("Notebook 30 bootstrap lower bound drift")
if not math.isclose(float(report["bootstrap_95"]["upper"]), EXPECTED_BOOTSTRAP_UPPER, abs_tol=1e-12):
    raise RuntimeError("Notebook 30 bootstrap upper bound drift")
print({
    "NB30_INPUT": "PASS",
    "verdict": complete["output_status"],
    "changed": len(changed),
    "rescued": len(rescued),
    "regressed": len(regressed),
})


## 4 — Nối query, ground truth và candidate B1


In [ ]:
nb29_complete_path = INPUT / "nb29-COMPLETE.json"
research.get(f"{nb29_root}/COMPLETE.json", nb29_complete_path)
if sha256_file(nb29_complete_path) != EXPECTED_NB29_COMPLETE_SHA256:
    raise RuntimeError("Notebook 29 COMPLETE SHA drift")
nb29 = json.loads(nb29_complete_path.read_text(encoding="utf-8"))
signed_payload(nb29, "manifest_payload_hash", canonical_hash)
worklist_spec = nb29["artifact_ledger"]["b1-worklist.jsonl"]
worklist_path = INPUT / "b1-worklist.jsonl"
research.get(f"{nb29_root}/b1-worklist.jsonl", worklist_path)
if worklist_path.stat().st_size != int(worklist_spec["bytes"]) or sha256_file(worklist_path) != worklist_spec["sha256"]:
    raise RuntimeError("Notebook 29 B1 worklist drift")
b1_rows = list(read_jsonl(worklist_path))
if len(b1_rows) != EXPECTED_B1_COUNT:
    raise RuntimeError("B1 worklist count drift")

queryset_path = REPO / "eval/queryset_dev.jsonl"
if sha256_file(queryset_path) != report["inputs"]["dev_queryset_sha256"]:
    raise RuntimeError("DEV queryset differs from notebook 30")
queries = {str(row["query_id"]): row for row in read_jsonl(queryset_path)}
if len(queries) != EXPECTED_DEV_QUERIES:
    raise RuntimeError("DEV query identity/count drift")

diagnostics = []
for score_row in changed:
    query_id = str(score_row["query_id"])
    query = queries[query_id]
    membership = candidate_memberships(query, b1_rows)
    diagnostics.append({
        **score_row,
        "direction": "RESCUED" if float(score_row["delta"]) > 0 else "REGRESSED",
        "author": query.get("author"),
        "query_text": query.get("text"),
        "gt_video_id": query["gt_video_id"],
        "wide_intervals": query_intervals(query, strict=False),
        "strict_intervals": query_intervals(query, strict=True),
        **membership,
        "causal_note": "Interval membership is supporting context, not proof that one listed frame caused the score change.",
    })
diagnostics.sort(key=lambda row: (-float(row["delta"]), str(row["query_id"])))
for row in diagnostics:
    print({
        "query_id": row["query_id"],
        "direction": row["direction"],
        "delta": row["delta"],
        "gt_video_id": row["gt_video_id"],
        "b1_in_wide_gt": len(row["b1_in_wide_gt"]),
        "b1_in_strict_gt": len(row["b1_in_strict_gt"]),
    }, flush=True)


## 5 — Xuất decision packet và publish


In [ ]:
decision = {
    "schema_version": 1,
    "status": "DIAGNOSTIC_COMPLETE",
    "decision": "PROMISING_BUT_INCONCLUSIVE",
    "decision_authority": "dev_diagnostic_only",
    "evidence": {
        "b0_mean": EXPECTED_B0_MEAN,
        "b1_mean": EXPECTED_B1_MEAN,
        "mean_delta": EXPECTED_MEAN_DELTA,
        "relative_uplift": EXPECTED_MEAN_DELTA / EXPECTED_B0_MEAN,
        "bootstrap_95": [EXPECTED_BOOTSTRAP_LOWER, EXPECTED_BOOTSTRAP_UPPER],
        "rescued_queries": EXPECTED_RESCUED,
        "regressed_queries": EXPECTED_REGRESSED,
        "unchanged_queries": EXPECTED_DEV_QUERIES - EXPECTED_CHANGED,
    },
    "budget": {
        "btc_unique_keyframes": 176707,
        "guardrail_ratio": 0.03,
        "formula": "floor(176707 * 0.03)",
        "locked_b1_keyframes": EXPECTED_B1_COUNT,
        "reason": "Pre-registered equal-budget guardrail shared by B1, B2, B3 and B4.",
    },
    "recommended_next": [
        "Report the positive but uncertain DEV result to the leader.",
        "Do not merge B1 into production from this evidence alone.",
        "Do not open held-out until a policy/decision is explicitly frozen.",
        "For stronger evidence, pre-register additional DEV queries or fund a full-feature equal-budget B4 control.",
    ],
    "not_recommended": "Rerunning deterministic notebook 30 does not increase the effective DEV sample size.",
    "nb29_complete_sha256": EXPECTED_NB29_COMPLETE_SHA256,
    "nb30_complete_sha256": sha256_file(complete_path),
    "diagnostic_count": len(diagnostics),
    "repo_commit": repo_commit,
    "heldout_accessed": False,
    "btc_mutated": False,
    "production_go_claimed": False,
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
}
decision["decision_payload_hash"] = canonical_hash(decision)
decision_path = write_json_atomic(OUTPUT / "decision.json", decision)
diagnostic_path = write_jsonl_atomic(OUTPUT / "changed-query-diagnostics.jsonl", diagnostics)

lines = [
    "# B1 DEV decision packet",
    "",
    "## Kết luận",
    "",
    "B1 tăng điểm trung bình trên DEV nhưng khoảng tin cậy 95% vẫn cắt qua 0. "
    "Kết quả là **PROMISING_BUT_INCONCLUSIVE**, chưa đủ để merge production.",
    "",
    "## Số đo",
    "",
    f"- B0 mean: `{EXPECTED_B0_MEAN:.6f}`",
    f"- B1 mean: `{EXPECTED_B1_MEAN:.6f}`",
    f"- Mean delta: `{EXPECTED_MEAN_DELTA:+.6f}`",
    f"- Bootstrap 95%: `[{EXPECTED_BOOTSTRAP_LOWER:.6f}, {EXPECTED_BOOTSTRAP_UPPER:.6f}]`",
    f"- Query: `{EXPECTED_RESCUED}` tăng, `{EXPECTED_REGRESSED}` giảm, `{EXPECTED_DEV_QUERIES - EXPECTED_CHANGED}` không đổi",
    "",
    "## Ngân sách keyframe",
    "",
    "BTC có 176.707 keyframe duy nhất. Guardrail 3% đã khóa trước cho phép "
    "`floor(176707 × 0.03) = 5301` keyframe bổ sung cho mỗi arm B1–B4.",
    "",
    "## Ba query thay đổi",
    "",
]
for row in diagnostics:
    lines.extend([
        f"### {row['query_id']} — {row['direction']}",
        "",
        f"- Delta: `{float(row['delta']):+.6f}`",
        f"- Query: {row['query_text']}",
        f"- Video GT: `{row['gt_video_id']}`",
        f"- B1 trong vùng rộng/chặt: `{len(row['b1_in_wide_gt'])}/{len(row['b1_in_strict_gt'])}`",
        "",
    ])
lines.extend([
    "## Hành động đề xuất",
    "",
    "Báo leader đây là tín hiệu dương nhưng chưa chắc chắn; giữ toàn bộ artifact research, "
    "chưa merge BTC và chưa mở held-out khi chưa khóa quyết định tiếp theo.",
    "",
])
handoff_path = OUTPUT / "leader-handoff.md"
handoff_path.write_text("\n".join(lines), encoding="utf-8")

artifact_ledger = {
    path.name: {"sha256": sha256_file(path), "bytes": path.stat().st_size}
    for path in (decision_path, diagnostic_path, handoff_path)
}
complete = {
    "schema_version": 1,
    "status": "COMPLETE",
    "output_status": "PROMISING_BUT_INCONCLUSIVE",
    "artifact_ledger": artifact_ledger,
    "artifact_ledger_sha256": canonical_hash(artifact_ledger),
    "heldout_accessed": False,
    "btc_mutated": False,
    "completed_at_utc": datetime.now(timezone.utc).isoformat(),
}
complete["manifest_payload_hash"] = canonical_hash(complete)
complete_path_out = write_json_atomic(OUTPUT / "COMPLETE.json", complete)

for path in (decision_path, diagnostic_path, handoff_path):
    remote = f"{output_root}/{path.name}"
    research.put(path, remote)
    if research.sha256(remote) != sha256_file(path):
        raise RuntimeError(f"Diagnostic upload verification failed: {path.name}")
complete_remote = f"{output_root}/COMPLETE.json"
research.put(complete_path_out, complete_remote)
if research.sha256(complete_remote) != sha256_file(complete_path_out):
    raise RuntimeError("Diagnostic COMPLETE upload verification failed")
print({
    "NOTEBOOK31": "COMPLETE",
    "decision": decision["decision"],
    "changed_queries": [row["query_id"] for row in diagnostics],
    "remote_output": output_root,
    "leader_handoff": f"{output_root}/leader-handoff.md",
    "next": "Send the handoff to the leader; do not merge or open held-out without an explicit frozen decision.",
})
